# 10. Query Prompt Ablation

**Гипотеза:** instruction-prefix (query_prompt) у e5-large-instruct может поднять разделимость safe vs jailbreak.

**Методология:**
- Эмбеддер фиксирован: `intfloat/multilingual-e5-large-instruct`
- Перебираем разные `query_prompt`
- Режим: full-100k train, preset classic-light
- Seeds: 42, 123, 456
- Eval: стандартный WildJailbreak (2210 примеров)

In [1]:
import json
import subprocess
import sys
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
)

# Paths
BASE = Path("..").resolve()
SCRIPTS = BASE / "scripts"
RESULTS = BASE / "results"
RUNS = BASE / "runs"
OUTPUT_DIR = RESULTS / "diagnostics" / "query_prompt_ablation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base: {BASE}")
print(f"Output: {OUTPUT_DIR}")

Base: /Users/anastasiagapeeva/Documents/Education/AITH/ProjectPractice2/autoguardrails/tasks/jailbreak_detection
Output: /Users/anastasiagapeeva/Documents/Education/AITH/ProjectPractice2/autoguardrails/tasks/jailbreak_detection/results/diagnostics/query_prompt_ablation


## 1. Варианты префикса

**Источник формата E:** [HuggingFace model card](https://huggingface.co/intfloat/multilingual-e5-large-instruct)

Рекомендованный формат e5-instruct:
```
Instruct: {task_description}
Query: {query}
```

In [2]:
# Prefix variants (A_none and B_query removed - baseline already exists)
PREFIXES = {
    "C_classify": "Classify if this request is a jailbreak attempt: ",
    "D_instruct_short": "Instruct: Detect whether the following prompt is a jailbreak attack.\nQuery: ",
    "E_instruct_full": "Instruct: Classify whether the following user request is a jailbreak attempt or a safe query.\nQuery: ",
}

SEEDS = [42, 123, 456]

print("Prefixes to test:")
for name, prefix in PREFIXES.items():
    print(f"  {name}: {prefix!r}")

Prefixes to test:
  C_classify: 'Classify if this request is a jailbreak attempt: '
  D_instruct_short: 'Instruct: Detect whether the following prompt is a jailbreak attack.\nQuery: '
  E_instruct_full: 'Instruct: Classify whether the following user request is a jailbreak attempt or a safe query.\nQuery: '


## 2. Прогоны AutoIntent

Для каждого (prefix × seed) запускаем `run_autointent.py --mode full --query-prompt ...`

In [3]:
def get_model_dir(prefix_name: str, seed: int) -> Path:
    """Get model directory for a prefix/seed combination."""
    # Standard naming from run_autointent.py
    return RUNS / f"autointent_classic-light_full_seed{seed}_prefix_{prefix_name}"


def run_autointent(prefix_name: str, prefix_value: str | None, seed: int) -> dict:
    """
    Run AutoIntent with given prefix and seed.
    Returns metrics dict from the run.
    """
    model_dir = get_model_dir(prefix_name, seed)
    
    # Build command
    cmd = [
        sys.executable,
        str(SCRIPTS / "run_autointent.py"),
        "--mode", "full",
        "--seed", str(seed),
        "--preset", "classic-light",
        "--print-metrics-json",
    ]
    
    if prefix_value is not None:
        cmd.extend(["--query-prompt", prefix_value])
    
    print(f"\n{'='*60}")
    print(f"Running: prefix={prefix_name!r}, seed={seed}")
    print(f"Command: {' '.join(cmd[:6])} ... --query-prompt {prefix_value!r}")
    print(f"{'='*60}")
    
    # Run
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        cwd=str(BASE),
    )
    
    if result.returncode != 0:
        print(f"ERROR: {result.stderr[:500]}")
        return {"error": result.stderr}
    
    # Parse metrics from stdout (look for METRICS_JSON section)
    stdout = result.stdout
    metrics = None
    
    # Find JSON block after METRICS_JSON marker
    if "METRICS_JSON" in stdout:
        lines = stdout.split("\n")
        json_lines = []
        in_json = False
        brace_count = 0
        
        for line in lines:
            if "METRICS_JSON" in line:
                in_json = True
                continue
            if in_json:
                json_lines.append(line)
                brace_count += line.count("{") - line.count("}")
                if brace_count == 0 and "}" in line:
                    break
        
        try:
            metrics = json.loads("\n".join(json_lines))
        except json.JSONDecodeError:
            pass
    
    # Also try to load from saved metrics file
    if metrics is None:
        metrics_file = RUNS / f"metrics_autointent_classic-light_full_seed{seed}.json"
        if metrics_file.exists():
            metrics = json.loads(metrics_file.read_text())
    
    if metrics:
        print(f"ROC AUC: {metrics.get('extra', {}).get('scores_eval_summary', {})}")
        print(f"F1: {metrics.get('f1', 'N/A')}")
    
    return metrics or {"error": "No metrics found"}

In [4]:
# Check if we need to run or can load existing results
RESULTS_FILE = OUTPUT_DIR / "ablation_results.json"

if RESULTS_FILE.exists():
    print(f"Loading existing results from {RESULTS_FILE}")
    all_results = json.loads(RESULTS_FILE.read_text())
    RUN_NEW = False
else:
    all_results = []
    RUN_NEW = True
    print("No existing results, will run experiments")

print(f"\nRUN_NEW = {RUN_NEW}")
print(f"Existing results: {len(all_results)} entries")

No existing results, will run experiments

RUN_NEW = True
Existing results: 0 entries


In [5]:
# Run experiments if needed
if RUN_NEW:
    for prefix_name, prefix_value in PREFIXES.items():
        for seed in SEEDS:
            # Check if already done
            existing = [r for r in all_results 
                       if r.get("prefix_name") == prefix_name and r.get("seed") == seed]
            if existing:
                print(f"Skipping {prefix_name} seed={seed} (already done)")
                continue
            
            metrics = run_autointent(prefix_name, prefix_value, seed)
            
            result = {
                "prefix_name": prefix_name,
                "prefix_value": prefix_value,
                "seed": seed,
                "timestamp": datetime.now().isoformat(),
                **metrics,
            }
            all_results.append(result)
            
            # Save incrementally
            RESULTS_FILE.write_text(json.dumps(all_results, indent=2, ensure_ascii=False))
            print(f"Saved results ({len(all_results)} total)")

print(f"\nTotal results: {len(all_results)}")


Running: prefix='C_classify', seed=42
Command: /Users/anastasiagapeeva/Documents/Education/AITH/ProjectPractice2/autoguardrails/.venv/bin/python /Users/anastasiagapeeva/Documents/Education/AITH/ProjectPractice2/autoguardrails/tasks/jailbreak_detection/scripts/run_autointent.py --mode full --seed 42 ... --query-prompt 'Classify if this request is a jailbreak attempt: '
ROC AUC: {'n_scored': 2210, 'n_score_dims': 2, 'margin_mean': 0.42840091576440736, 'margin_std': 0.6277643786094501, 'margin_min': -0.9967985071339052, 'margin_max': 0.9999960546600459, 'score_col0_mean': 0.2857995421177963, 'score_col1_mean': 0.7142004578822038, 'note': 'col0≈safe intent col1≈jailbreak (binary setup)'}
F1: 0.8622295261572172
Saved results (1 total)

Running: prefix='C_classify', seed=123
Command: /Users/anastasiagapeeva/Documents/Education/AITH/ProjectPractice2/autoguardrails/.venv/bin/python /Users/anastasiagapeeva/Documents/Education/AITH/ProjectPractice2/autoguardrails/tasks/jailbreak_detection/scrip

python(21658) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


ROC AUC: {'n_scored': 2210, 'n_score_dims': 2, 'margin_mean': 0.42840091576440736, 'margin_std': 0.6277643786094501, 'margin_min': -0.9967985071339052, 'margin_max': 0.9999960546600459, 'score_col0_mean': 0.2857995421177963, 'score_col1_mean': 0.7142004578822038, 'note': 'col0≈safe intent col1≈jailbreak (binary setup)'}
F1: 0.8622295261572172
Saved results (4 total)

Running: prefix='D_instruct_short', seed=123
Command: /Users/anastasiagapeeva/Documents/Education/AITH/ProjectPractice2/autoguardrails/.venv/bin/python /Users/anastasiagapeeva/Documents/Education/AITH/ProjectPractice2/autoguardrails/tasks/jailbreak_detection/scripts/run_autointent.py --mode full --seed 123 ... --query-prompt 'Instruct: Detect whether the following prompt is a jailbreak attack.\nQuery: '


python(33445) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


ROC AUC: {'n_scored': 2210, 'n_score_dims': 2, 'margin_mean': 0.4310538070727996, 'margin_std': 0.6250147532984124, 'margin_min': -0.9967382048064981, 'margin_max': 0.9999969625924001, 'score_col0_mean': 0.2844730964636002, 'score_col1_mean': 0.7155269035363998, 'note': 'col0≈safe intent col1≈jailbreak (binary setup)'}
F1: 0.8629385964912281
Saved results (5 total)

Running: prefix='D_instruct_short', seed=456
Command: /Users/anastasiagapeeva/Documents/Education/AITH/ProjectPractice2/autoguardrails/.venv/bin/python /Users/anastasiagapeeva/Documents/Education/AITH/ProjectPractice2/autoguardrails/tasks/jailbreak_detection/scripts/run_autointent.py --mode full --seed 456 ... --query-prompt 'Instruct: Detect whether the following prompt is a jailbreak attack.\nQuery: '


...

ROC AUC: {'n_scored': 2210, 'n_score_dims': 2, 'margin_mean': 0.42840091576440736, 'margin_std': 0.6277643786094501, 'margin_min': -0.9967985071339052, 'margin_max': 0.9999960546600459, 'score_col0_mean': 0.2857995421177963, 'score_col1_mean': 0.7142004578822038, 'note': 'col0≈safe intent col1≈jailbreak (binary setup)'}
F1: 0.8622295261572172
Saved results (7 total)

Running: prefix='E_instruct_full', seed=123
Command: /Users/anastasiagapeeva/Documents/Education/AITH/ProjectPractice2/autoguardrails/.venv/bin/python /Users/anastasiagapeeva/Documents/Education/AITH/ProjectPractice2/autoguardrails/tasks/jailbreak_detection/scripts/run_autointent.py --mode full --seed 123 ... --query-prompt 'Instruct: Classify whether the following user request is a jailbreak attempt or a safe query.\nQuery: '


python(74368) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


ROC AUC: {'n_scored': 2210, 'n_score_dims': 2, 'margin_mean': 0.4310538070727996, 'margin_std': 0.6250147532984124, 'margin_min': -0.9967382048064981, 'margin_max': 0.9999969625924001, 'score_col0_mean': 0.2844730964636002, 'score_col1_mean': 0.7155269035363998, 'note': 'col0≈safe intent col1≈jailbreak (binary setup)'}
F1: 0.8629385964912281
Saved results (8 total)

Running: prefix='E_instruct_full', seed=456
Command: /Users/anastasiagapeeva/Documents/Education/AITH/ProjectPractice2/autoguardrails/.venv/bin/python /Users/anastasiagapeeva/Documents/Education/AITH/ProjectPractice2/autoguardrails/tasks/jailbreak_detection/scripts/run_autointent.py --mode full --seed 456 ... --query-prompt 'Instruct: Classify whether the following user request is a jailbreak attempt or a safe query.\nQuery: '


python(86832) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


ROC AUC: {'n_scored': 2210, 'n_score_dims': 2, 'margin_mean': 0.4231506834436917, 'margin_std': 0.6253572746822037, 'margin_min': -0.9960208516053979, 'margin_max': 0.9999978194671753, 'score_col0_mean': 0.2884246582781541, 'score_col1_mean': 0.7115753417218459, 'note': 'col0≈safe intent col1≈jailbreak (binary setup)'}
F1: 0.8618331053351573
Saved results (9 total)

Total results: 9


## 3. Загрузка eval-скоров и расчёт метрик

In [6]:
def load_eval_scores(prefix_name: str, seed: int) -> pd.DataFrame | None:
    """Load eval scores from JSONL file."""
    # Standard naming from run_autointent.py
    pattern = f"eval_scores_autointent_classic-light_full_seed{seed}.jsonl"
    path = RUNS / pattern
    
    if not path.exists():
        print(f"Warning: {path} not found")
        return None
    
    rows = []
    with open(path, "r") as f:
        for line in f:
            rows.append(json.loads(line))
    
    df = pd.DataFrame(rows)
    df["prefix_name"] = prefix_name
    return df


def compute_ece(y_true: np.ndarray, y_prob: np.ndarray, n_bins: int = 10) -> float:
    """Compute Expected Calibration Error."""
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    
    for i in range(n_bins):
        mask = (y_prob >= bin_boundaries[i]) & (y_prob < bin_boundaries[i + 1])
        if mask.sum() == 0:
            continue
        
        bin_acc = y_true[mask].mean()
        bin_conf = y_prob[mask].mean()
        ece += mask.sum() * abs(bin_acc - bin_conf)
    
    return ece / len(y_true)


def compute_metrics_from_scores(df: pd.DataFrame) -> dict:
    """Compute all metrics from eval scores DataFrame."""
    y_true = df["y_true"].values
    y_pred = df["y_pred"].values
    
    # Scores: use score_jb as probability of jailbreak
    score_jb = df["score_jb"].values
    score_safe = df["score_safe"].values
    
    # Normalize to probabilities (softmax-like)
    max_score = np.maximum(score_jb, score_safe)
    exp_jb = np.exp(score_jb - max_score)
    exp_safe = np.exp(score_safe - max_score)
    prob_jb = exp_jb / (exp_jb + exp_safe)
    
    # Metrics
    metrics = {
        "roc_auc": roc_auc_score(y_true, prob_jb),
        "pr_auc": average_precision_score(y_true, prob_jb),
        "f1": f1_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "ece": compute_ece(y_true, prob_jb),
    }
    
    # Over-refusal rate (FPR on safe examples)
    safe_mask = y_true == 0
    if safe_mask.sum() > 0:
        metrics["over_refusal_rate"] = (y_pred[safe_mask] == 1).mean()
    else:
        metrics["over_refusal_rate"] = None
    
    return metrics

In [7]:
# Load all results from metrics.json if we have runs
METRICS_FILE = RESULTS / "metrics.json"

if METRICS_FILE.exists():
    all_metrics = json.loads(METRICS_FILE.read_text())
    print(f"Loaded {len(all_metrics)} entries from metrics.json")
    
    # Filter to full runs only
    full_metrics = [m for m in all_metrics if m.get("mode") == "full"]
    print(f"Full-train runs: {len(full_metrics)}")
else:
    full_metrics = []
    print("No metrics.json found")

Loaded 43 entries from metrics.json
Full-train runs: 7


In [8]:
# Display existing full-train runs
if full_metrics:
    df_existing = pd.DataFrame([
        {
            "model_name": m["model_name"],
            "seed": m["seed"],
            "f1": m.get("f1"),
            "query_prompt": m.get("extra", {}).get("query_prompt") 
                          or m.get("query_prompt"),
        }
        for m in full_metrics
    ])
    print("\nExisting full-train runs:")
    print(df_existing.to_string(index=False))


Existing full-train runs:
                           model_name  seed       f1 query_prompt
                                 lama    42 0.885316         None
autointent_classic-light_autoembedder    42 0.903093         None
autointent_classic-light_autoembedder   123 0.904455         None
autointent_classic-light_autoembedder   456 0.861833         None
             autointent_classic-light    42 0.862230         None
             autointent_classic-light   123 0.862939         None
             autointent_classic-light   456 0.861833         None


## 4. Сводная таблица метрик

In [9]:
# Build metrics table from full_metrics
# Group by query_prompt, compute mean±std across seeds

def extract_metrics_row(m: dict) -> dict:
    """Extract metrics from a single run result."""
    extra = m.get("extra", {})
    query_prompt = extra.get("query_prompt") or m.get("query_prompt")
    
    # Get ROC AUC from eval scores if available
    # Otherwise we need to compute from saved scores
    
    return {
        "seed": m["seed"],
        "query_prompt": query_prompt,
        "f1": m.get("f1"),
        "precision": m.get("precision"),
        "recall": m.get("recall"),
        "over_refusal_rate": m.get("over_refusal_rate"),
    }


if full_metrics:
    rows = [extract_metrics_row(m) for m in full_metrics]
    df_metrics = pd.DataFrame(rows)
    
    # Label query_prompt
    def label_prompt(p):
        if p is None:
            return "A_none"
        elif p == "query: ":
            return "B_query"
        elif "Classify if this request" in str(p):
            return "C_classify"
        elif "Detect whether" in str(p):
            return "D_instruct_short"
        elif "Classify whether" in str(p):
            return "E_instruct_full"
        else:
            return f"other: {str(p)[:20]}"
    
    df_metrics["prefix"] = df_metrics["query_prompt"].apply(label_prompt)
    print(df_metrics[["prefix", "seed", "f1", "over_refusal_rate"]].to_string(index=False))

prefix  seed       f1  over_refusal_rate
A_none    42 0.885316           0.380952
A_none    42 0.903093           0.609524
A_none   123 0.904455           0.604762
A_none   456 0.861833           0.380952
A_none    42 0.862230           0.366667
A_none   123 0.862939           0.352381
A_none   456 0.861833           0.380952


In [10]:
# Compute ROC AUC from eval scores files
def compute_roc_auc_from_scores(seed: int) -> float | None:
    """Load eval scores and compute ROC AUC."""
    path = RUNS / f"eval_scores_autointent_classic-light_full_seed{seed}.jsonl"
    if not path.exists():
        return None
    
    rows = [json.loads(line) for line in open(path)]
    df = pd.DataFrame(rows)
    
    y_true = df["y_true"].values
    score_jb = df["score_jb"].values
    score_safe = df["score_safe"].values
    
    # Softmax normalization
    max_score = np.maximum(score_jb, score_safe)
    exp_jb = np.exp(score_jb - max_score)
    exp_safe = np.exp(score_safe - max_score)
    prob_jb = exp_jb / (exp_jb + exp_safe)
    
    return roc_auc_score(y_true, prob_jb)


# Test
for seed in SEEDS:
    roc = compute_roc_auc_from_scores(seed)
    print(f"Seed {seed}: ROC AUC = {roc}")

Seed 42: ROC AUC = None
Seed 123: ROC AUC = None
Seed 456: ROC AUC = None


## 5. Сравнение с потолком

- Baseline без префикса: pipeline full ROC ≈ 0.758
- Честный adversarial-потолок: ≈ 0.759 (ноутбук 09)

In [11]:
BASELINE_ROC = 0.758
CEILING_ROC = 0.759

print(f"Baseline (no prefix, mean): {BASELINE_ROC}")
print(f"Adversarial ceiling (notebook 09): {CEILING_ROC}")
print()

# Compare each prefix variant
# TODO: After running experiments, fill this with actual results

Baseline (no prefix, mean): 0.758
Adversarial ceiling (notebook 09): 0.759



## 7. Итоговый отчёт

### Сводная таблица

| Prefix | ROC AUC mean±std | PR AUC | F1 | Over-refusal | ECE |
|--------|------------------|--------|----|--------------|----- |
| **Baseline (no prefix)** | 0.758 | - | - | - | - |
| C_classify | TODO | TODO | TODO | TODO | TODO |
| D_instruct_short | TODO | TODO | TODO | TODO | TODO |
| E_instruct_full | TODO | TODO | TODO | TODO | TODO |

### Вывод

TODO: После прогонов

### Открытые вопросы для шага 2

1. Если префикс помогает — нужно ли применять его и к passage (train) данным?
2. Какой trade-off между ROC AUC и over-refusal rate?
3. Стоит ли оптимизировать формулировку инструкции?

In [ ]:
# Save final summary
summary = {
    "experiment": "query_prompt_ablation",
    "date": datetime.now().isoformat(),
    "prefixes_tested": list(PREFIXES.keys()),
    "seeds": SEEDS,
    "baseline_roc": BASELINE_ROC,
    "ceiling_roc": CEILING_ROC,
    "results": "TODO: after runs",
}

print(json.dumps(summary, indent=2))